[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/14_business_ai/44_architecture_patterns.ipynb)

# 📓 Notebook 44 — Architecture Patterns for AI Applications

> **Module:** Business AI in Practice · **Estimated time:** ~1.5 h · **Difficulty:** Intermediate

Once you know *what* to automate (NB 43), the next question is *how to structure it*. This notebook walks through four canonical architectures in order of growing complexity, plus the dedicated **end-to-end ML pipeline** pattern. Each one has a specific moment in an organisation's life when it's the right answer — and a specific failure mode when it's the wrong one.

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Draw and label** each of the five canonical architectures (single-tier, 3-tier, service-oriented, microservices, ML pipeline).
2. **Map a given AI requirement** to the simplest architecture that meets it.
3. **Recognise the smell** that a project has out-grown its current architecture.
4. **Avoid the two opposite mistakes**: over-engineering (microservices for a 5-user pilot) and under-engineering (a 200-user system held together by one script + cron).
5. **Identify the ML-specific concerns** (training data, model versioning, drift) that pure-software architecture patterns don't address.

**Prerequisites:** Module 11 (production) — packaging and scheduling. Strongly recommended: Modules 4 (ML) and 5 (AI engineering) so the examples land.

**Time budget:** ~75 minutes including diagrams + exercises.


## 1. 🧠 Mental model — architecture as the answer to a *traffic* question

The single most useful heuristic for choosing an architecture is: **what shape of traffic do you need to serve, and from whom?**

```
   1 user    │   10 users   │   100 users   │   1,000+ users
  one place  │   one team   │   one company │   many companies / public
  ──────────────────────────────────────────────────────────────────
             │              │               │
  Single-    │   3-tier     │   Service-    │   Microservices
  tier       │   (client-   │   oriented    │   (one team
  script     │    server)   │   (a few      │    per service)
             │              │    services,  │
             │              │    one team)  │
             │              │               │
             │              │   ML pipeline as a cross-cutting concern at any scale
```

The arrow goes from left to right as the user base, the request volume, or the *team* size grows. Crossing into the next column always brings new costs (deployment, monitoring, coordination). The key skill is not *building* the right one — it's choosing the right one for *now* and knowing the signs that you've outgrown it.


## 2. Pattern 1 — Single-tier (the script)

```
  ┌────────────────────────────────────────┐
  │       my_analysis.py / notebook.ipynb  │
  │                                         │
  │   read CSV  →  do work  →  print/save  │
  └────────────────────────────────────────┘
             ▲
             │
        one developer,
       run on their laptop
```

**Best for:** experimentation, one-off analysis, the first version of every project, anything where the user is *also* the developer.

**What you give up:** no separation of concerns; a second developer working on the same file conflicts immediately; no shared state between runs; redeployment = "send the new file."

**Smell of having outgrown it:** more than one human needs to invoke the script, or someone says *"can we run this nightly?"* (→ go to scheduled job, NB 40), or someone says *"can we expose this to a non-technical user?"* (→ go to 3-tier).

Every AI initiative starts here. The course up to NB 42 has been almost entirely single-tier — Jupyter notebooks doing analysis or producing a small artefact. That's the right starting point. Promoting prematurely is one of the most common over-engineering mistakes.


## 3. Pattern 2 — Three-tier (presentation / business logic / data)

```
  ┌──────────────────────┐        ┌──────────────────────┐        ┌──────────────────────┐
  │   PRESENTATION       │        │   BUSINESS LOGIC     │        │   DATA               │
  │                      │  HTTP  │                      │  SQL   │                      │
  │   Streamlit  /       │ ────▶  │  FastAPI / Flask /   │ ────▶  │  PostgreSQL /        │
  │   browser UI /       │        │  Python server       │        │  SQLite /            │
  │   thin React         │ ◀────  │  (your domain code)  │ ◀────  │  S3 / blob store     │
  │                      │  JSON  │                      │  rows  │                      │
  └──────────────────────┘        └──────────────────────┘        └──────────────────────┘
        users see UI                  *your* code lives here          state lives here
```

**Best for:** the first system that needs to be used by people who aren't the developer. The classic shape for a small internal tool — an admin panel, an internal dashboard, a customer-support back-office form.

**AI fits naturally** in the middle tier: a `POST /classify` endpoint that calls an LLM and writes the result to the DB; a Streamlit chat UI that calls the middle tier. The LLM API is *external* — you don't run the model, you call it.

**Smell of having outgrown it:** the middle tier accumulates unrelated concerns (auth + LLM + dashboards + scheduled jobs + the OCR pipeline + a webhook handler …) and deploying any one of them requires re-deploying all of them. That's the signal to break it up.


## 4. Pattern 3 — Service-oriented (a few services, one team)

```
                                  ┌─────────────────────┐
                                  │  API gateway / BFF  │
                                  │  (one entry point)  │
                                  └────────┬────────────┘
                  ┌────────────────────────┼────────────────────────┐
                  ▼                        ▼                        ▼
         ┌────────────────┐      ┌─────────────────┐       ┌─────────────────┐
         │ classify-svc   │      │  rag-svc        │       │  user-svc       │
         │ (LLM router)   │      │  (vector store) │       │  (auth, profile)│
         └───────┬────────┘      └───────┬─────────┘       └────────┬────────┘
                 │                       │                          │
                 ▼                       ▼                          ▼
         ┌────────────────────────────────────────────────────────────────┐
         │             shared infrastructure (DB, observability)          │
         └────────────────────────────────────────────────────────────────┘
```

**Best for:** the second AI feature. You have one team but multiple distinct *capabilities* (classification, retrieval, user-management) — each becomes its own service, all behind a single gateway.

**Why it's a step up from 3-tier:** redeploying the classifier doesn't require redeploying the user-auth service. Failures are isolated. Each service can be scaled independently. A new feature is *another service*, not another route in a monolithic file.

**Smell of having outgrown it:** services start *needing* each other in ways that aren't a simple API call — service A's deploy requires coordinating with service B's deploy because they share a schema. The shared infrastructure becomes a bottleneck (everyone hits the same DB). One team is now too small to own all of it.

> 💡 **A common misuse:** treating *every Python module* as a service. The right unit is a *capability* (something a customer or another service would meaningfully call). Five capabilities is fine; fifty is over-engineering.


### 🔬 What actually happens when one component fails? — *failure isolation*, demonstrated

Sections 3–5 keep repeating the same headline reason to climb a tier: *"failures are isolated."* That sentence is easy to nod at and hard to *feel*. So let's stop asserting it and **run it**. Below is the smallest honest offline model of two architectures handling the same request — a single fault injected into each — so you can watch the *blast radius* differ.

A user request in our toy app needs three capabilities: **auth** (who are you?), **classify** (an LLM call — the flaky part), and **profile** (your saved settings). We'll inject the *same* failure — `classify` throws — into two designs and compare what the user gets back.

```text
   MONOLITH (one process, one try)          SERVICE-ORIENTED (gateway + isolated calls)
   ────────────────────────────────         ──────────────────────────────────────────
   handle(request):                         gateway(request):
     auth()        ✓                          auth-svc        ✓   (its own try/except)
     classify()    ✗  ← throws                classify-svc    ✗   ← caught → fallback
     profile()     ── never runs              profile-svc     ✓   (still runs)
        │                                          │
        ▼                                          ▼
   WHOLE request fails (502)                 DEGRADED but useful response
```

The mechanic is not magic: it's a **try/except boundary drawn around each capability**, plus the cost of getting there — an indirection (a "network hop" in real life). We'll demonstrate both the benefit *and* the cost, because the notebook's whole thesis is that the step up is a trade, not a free win.


In [ ]:
# Offline simulation — stdlib only, deterministic. No network, no LLM.
# Three capabilities. `classify` is the flaky one: we force it to fail.

def auth(req):
    return {"user": req["user"], "role": "agent"}

def classify(req):
    # The flaky capability. In real life: an LLM/API call. Here: a forced fault.
    raise RuntimeError("classify-svc unavailable (simulated outage)")

def profile(req):
    return {"theme": "dark", "signature": f"-- {req['user']}"}


# ─── Design A: MONOLITH ───────────────────────────────────────────────
# One process, one try-block around the whole request. Any failure kills it all.
def handle_monolith(req):
    a = auth(req)
    c = classify(req)          # ← throws here...
    p = profile(req)           # ← ...so this line NEVER runs
    return {"auth": a, "classify": c, "profile": p}


# ─── Design B: SERVICE-ORIENTED ───────────────────────────────────────
# The gateway calls each capability behind its OWN boundary. One failing
# service degrades to a fallback; the others still return real data.
def call_service(name, fn, req, fallback):
    try:
        return fn(req), "ok"
    except Exception as e:
        return fallback, f"DEGRADED ({e})"   # graceful degradation

def handle_service_oriented(req):
    a, a_status = call_service("auth",     auth,     req, fallback=None)
    c, c_status = call_service("classify", classify, req, fallback={"topic": "uncategorised"})
    p, p_status = call_service("profile",  profile,  req, fallback=None)
    return {
        "auth": a, "classify": c, "profile": p,
        "_status": {"auth": a_status, "classify": c_status, "profile": p_status},
    }


request = {"user": "alice"}

print("── MONOLITH ──")
try:
    print(handle_monolith(request))
except Exception as e:
    print(f"502 WHOLE REQUEST FAILED → {type(e).__name__}: {e}")
    print("blast radius: 3/3 capabilities lost (auth + profile were collateral damage)\n")

print("── SERVICE-ORIENTED ──")
result = handle_service_oriented(request)
for cap, status in result["_status"].items():
    print(f"  {cap:9} {status}")
print("blast radius: 1/3 capabilities degraded; auth + profile served real data")
print("response still usable:", {k: v for k, v in result.items() if k != "_status"})


In [ ]:
# But isolation is NOT free. The notebook is honest that climbing a tier is a
# TRADE. Here is the cost the diagram in NB 32 names: the indirection / "network hop".
# We make it visible by counting the hops each request pays for.

hops = {"count": 0}

def with_hop(fn):
    """Wrap a capability so calling it across a service boundary is counted."""
    def wrapper(req):
        hops["count"] += 1          # in real life: a real network round-trip (latency + a thing that can fail)
        return fn(req)
    return wrapper

def working_classify(req):
    return {"topic": "billing"}     # the happy path, so we measure cost, not failure

# Monolith: 3 plain in-process function calls — 0 network hops.
hops["count"] = 0
auth(request); working_classify(request); profile(request)
monolith_hops = hops["count"]

# Service-oriented: every capability is a separate service → 1 hop each.
hops["count"] = 0
svc_auth     = with_hop(auth)
svc_classify = with_hop(working_classify)
svc_profile  = with_hop(profile)
svc_auth(request); svc_classify(request); svc_profile(request)
service_hops = hops["count"]

print(f"monolith network hops per request:        {monolith_hops}")
print(f"service-oriented network hops per request: {service_hops}")
print()
print("Each hop is latency you didn't have before AND a new place that can fail —")
print("which is exactly why you add the try/except boundaries from the cell above.")
print("Isolation BUYS resilience; it PAYS in hops, health checks, and graceful degradation.")


> 🧠 **The takeaway the diagrams can't give you.** The phrase *"failures are isolated"* is just shorthand for **"every capability sits behind its own try/except (a service boundary), so one failing capability degrades to a fallback instead of taking the whole request down."** That's the entire mechanic — and you can now see it run.
>
> But the second cell is the half learners skip: that boundary **costs a hop**. A monolith pays 0 hops and fails all-or-nothing; the service split pays 1 hop per capability and fails gracefully. Neither is "better" in the abstract — which is the whole point of Sections 6–7. You step up a tier **when the cost of an all-or-nothing failure exceeds the cost of the hops**, and not one feature sooner.
>
> 🎯 This is the runnable kernel underneath every box-and-arrow diagram in this notebook: a *capability boundary* is a `try/except` around a call you could have made in-process — trading latency and operational overhead for the ability to degrade instead of crash. NB 28 §7 builds exactly this split for real, against a live model service.


## 5. Pattern 4 — Microservices (one team per service)

```
                                  ┌─────────────────────┐
                                  │  API gateway        │
                                  └────────┬────────────┘
         ┌────────┬────────┬───────────────┼────────────┬──────────┬──────────┐
         ▼        ▼        ▼               ▼            ▼          ▼          ▼
      svc-1   svc-2    svc-3            svc-4         svc-5    svc-N      svc-N+1
        │       │        │                │             │        │           │
        ▼       ▼        ▼                ▼             ▼        ▼           ▼
       db-1   db-2     db-3             db-4          db-5      db-N      …each owned by
                                                                          a different team

       Each service:
         • owns its data store          • independent CI/CD
         • independent scaling          • communication via events / queues / async messaging
         • independent on-call rotation
```

**Best for:** large engineering organisations (multiple teams, often 100+ engineers) where coordination cost between teams would otherwise dominate. Each team owns one or more services end-to-end.

**Why this is *not* the default:** the coordination cost shifts from "between people" to "between systems." You need: a service mesh or API gateway, a message bus or event log, distributed tracing, per-service on-call. A 5-engineer startup running 30 microservices typically has fewer outages going back to a monolith.

**Smell of having mis-deployed it:** every "new feature" requires changes across 4+ services, your end-to-end tests are flaky for cross-service reasons, and developers spend more time on service-to-service contracts than on user-visible work.

> ⚠️ **The over-engineering trap.** Microservices became fashionable around 2015 and many teams adopted them *before* they had the team size and operational maturity to support them. The 2020s correction is *modular monoliths* (one deployable, but cleanly separated modules) as a happy middle ground for most teams.

> 🔭 **You will actually make this cut.** NB 28 §7 splits the 3-tier POC into a gateway + model microservice — the smallest honest version of this pattern, including what it costs (a network hop, a second health check, graceful degradation).


## 6. The cross-cutting pattern — end-to-end ML / AI pipeline

ML pipelines deserve their own pattern because they don't slot into the *request-response* shape the previous four patterns serve. They handle the *data → model → deployed-model → monitoring* lifecycle.

```
  ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐
  │  INGEST    │──▶│  PREPARE   │──▶│  TRAIN     │──▶│  EVALUATE  │──▶│  SERVE     │
  │  data from │   │  clean,    │   │  fit a     │   │  on hold-  │   │  inference │
  │  sources   │   │  feature   │   │  model     │   │  out + on  │   │  endpoint  │
  │            │   │  engineer  │   │            │   │  shadow    │   │            │
  └────────────┘   └────────────┘   └────────────┘   └────────────┘   └─────┬──────┘
                                                                              │
                                                                              ▼
                                                                       ┌────────────┐
                                                                       │  MONITOR   │
                                                                       │  drift,    │
                                                                       │  cost,     │
                                                                       │  latency   │
                                                                       │            │──▶ feeds
                                                                       │            │    back to
                                                                       └────────────┘    INGEST
```

Each stage is itself a service or a scheduled job. The *artifact* that moves between them is **the model**, not just data: trained models are versioned, signed, and shipped through the same kind of CI/CD a software service uses.

**What's specific to ML/AI** (beyond regular software pipelines):

- **Training data is part of the artefact.** Two models that look identical but were trained on different snapshots are *different products*.
- **Drift exists.** A model that was 92% accurate in March will not necessarily be 92% accurate in October. Monitoring has to compare against ground-truth labels that arrive *days later*.
- **The feedback loop has economic cost.** Every prediction your model makes nudges the world; the next training-data batch reflects the nudge. This is real for recommender systems, classifiers, RAG bots.

Notebooks 25 (evaluation & observability) and 23 (scheduling) are the technical foundation; this notebook is the *architectural* framing.


## 7. The right size for *your* AI project — a decision table

| If your project looks like this… | The simplest workable architecture is… |
|---|---|
| One analyst doing exploratory work on a CSV | **Single-tier** (Jupyter notebook). Don't over-engineer. |
| One feature, used by < 100 people inside one company, runs daily | **Single-tier scheduled job** (cron + a script). NB 39 + NB 40 are the template. |
| One feature, used interactively by employees who are not developers | **3-tier** (Streamlit/FastAPI/DB). NB 42 capstone is close to this. |
| Two or three distinct AI capabilities, one team owns all of them | **Service-oriented** with shared infrastructure. |
| 4+ teams, each shipping at their own cadence, multiple AI features in production | **Microservices** — but only if you also have the platform team to support it. |
| Any of the above, plus a model you trained yourself | **Add an ML pipeline** as a cross-cutting concern. |

> 🎯 **The rule.** *Pick the architecture you'll have outgrown in 12 months, not the one you'll need in 36.* Migrations are real but rarely worse than years of carrying complexity you didn't need.


## 8. How the course's notebooks map to these patterns

If you've worked through the course, you have already touched most of these:

| Pattern | Where it appears in the course |
|---|---|
| Single-tier (script/notebook) | Every notebook from NB 1 to NB 42, by default |
| Scheduled single-tier | NB 40 — scheduling & orchestration |
| 3-tier with AI in the middle | NB 42 capstone (the AI assistant ships as a small service skeleton) |
| Service-oriented | Implicit in `llm_providers.py` — pluggable providers, common interface |
| Microservices | Not built — too big for a teaching course; the *patterns* live here |
| ML pipeline | NB 14–16 + NB 40 — the train/evaluate/serve loop |

The exercises below ask you to *decide* which pattern fits a given new project, not to *build* the bigger ones. Architecture decisions are made on a whiteboard, with the team — that's the skill to develop in this notebook.


## 🧪 Practice exercises

Reflection / design exercises. Each asks you to *choose* an architecture and *defend* the choice.

### Exercise 1 — ⭐ Pick the right pattern

A finance team wants a daily report that classifies the previous day's customer support tickets by topic and emails a summary to the head of support. There are ~300 tickets per day. No interactive interface needed; nobody else needs the output. Which pattern? Defend the choice in one paragraph.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Single-tier scheduled job.** This is the canonical *one-script-run-by-cron* case. Inputs come from one source (the ticket DB), output goes to one channel (email), there's no interactive user, no multi-user concurrency, and the volume (300/day) is trivial. Building this as a 3-tier system would add 80% effort and zero value. The script lives in a package (NB 39), cron triggers it nightly (NB 40), failures alert via email — done.
</details>

### Exercise 2 — ⭐⭐ Recognise the outgrowth signal

A 3-tier Streamlit app for ticket triage has been running for six months. The PM now wants to add (a) a webhook that triggers when a new ticket arrives in the queue, (b) a nightly report, and (c) a separate admin panel for adjusting the rules. The engineering team is one developer. What's happening, and what would you do?

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**The middle tier is sprouting unrelated responsibilities** (webhook handler + scheduled job + admin panel + the original UI), which is the classic 3-tier outgrowth smell. With one developer, the right move is **not** microservices — the coordination cost would dominate. Instead: keep the deploy as a single 3-tier monolith for now, but *modularise internally*: separate Python packages for `webhook/`, `nightly/`, `admin/`, `triage/`, each tested independently. This is the *modular monolith* — you get most of the separation benefits without the operational cost of multiple deploys. If a second developer joins and the deploys start blocking each other, *that's* the signal to split into services.
</details>

### Exercise 3 — ⭐⭐ ML-pipeline awareness

Your team built a churn-prediction model six months ago. It was 87% accurate on the test set. Sales reports a wave of customer complaints that the model is calling loyal customers "likely to churn," which led to a poorly-targeted retention campaign. What architectural component was missing, and what would you add?

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Monitoring was missing.** The team treated the model as a static artefact deployed once; what's actually happening is the input distribution has shifted (perhaps a new pricing tier was launched, or the marketing-attribution code changed how a customer is recorded). What to add (in priority order): (1) a *drift dashboard* comparing the live input distribution against the training-set distribution per-feature, weekly; (2) a *prediction-vs-outcome* dashboard that waits 30 days for actual churn outcomes and computes live accuracy; (3) a *shadow-model* deploy of the next candidate so you can compare new vs. old before promoting. This is the *MONITOR* node of the ML pipeline diagram — the team built a one-shot inference service when they needed a pipeline.
</details>

### Exercise 4 — ⭐⭐ The over-engineering trap

A 6-person startup pitches you their architecture: 12 microservices, a service mesh (Istio), a Kubernetes cluster, separate Postgres + Redis + Kafka instances per environment, distributed tracing via Jaeger. They have 80 users. What concerns would you raise, and what would you suggest instead?

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**The architecture is sized for an organisation 30× larger than the team running it.** Concerns: (1) at 6 people, every developer is on-call for multiple services they didn't write; (2) operational complexity (mesh, K8s, distributed tracing) will consume more engineer-hours than user-facing work; (3) the team almost certainly cannot afford to keep all of it healthy *and* ship features at competitive speed; (4) when a single user-facing bug requires touching 4 services, root-cause analysis becomes a week-long expedition.

**Suggested alternative:** a modular monolith — one Python project with cleanly-separated packages, one Postgres database, one deployment, one CI/CD pipeline. Add complexity only when the team has *proof* (not anticipation) of the bottleneck the complexity solves. The Stripe / Shopify / Basecamp engineering blogs all have well-known articles on this; the 2020s consensus is that *most* product companies are better off as long as they can be as modular monoliths.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞 (architecture-review)

Read the following architecture decision-record (ADR) excerpt and find at least three things that should make a reviewer push back.

> *"We will build the new AI customer-support feature as a Python microservice using FastAPI. It will call OpenAI's GPT-4o for classification and write results to a new dedicated Postgres database. Authentication will be handled by a new auth microservice we'll build alongside it. We expect ~50 tickets/day initially, growing to ~500 by year-end. Time to MVP: 2 weeks."*

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**1. Over-architected for volume.** 50 tickets/day → 0.0006 req/s. A microservice with its own DB is wildly oversized; a Streamlit + SQLite + one cron is more honest. The reviewer should ask: *"do you need this to scale to 500/day, or to 50,000/day?"*

**2. Building a new auth service in 2 weeks alongside the feature is a red flag.** Authentication should never be a side project. Either reuse the existing company auth (SSO / OAuth proxy) or accept that auth is itself a multi-month investment.

**3. Single-provider lock-in.** Calling `OpenAI's GPT-4o` directly in the code makes provider swap a refactor. The course's `llm_providers.py` shim is the right pattern — same interface across providers.

**4. No mention of evaluation, monitoring, or fallback.** What happens if OpenAI is down? What's the success metric? Who's on call? A 2-week MVP that skips all three of those is a 6-week production incident waiting to happen.

**5. Time estimate is ambitious.** New service + new DB + new auth + LLM integration in 2 weeks, with no operational concerns budgeted? Add a factor of 2–3.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Sketch a 3-tier AI feature

Sketch (as an ASCII or text diagram) the architecture of an internal HR-FAQ chatbot for a 200-employee company. Constraints: must use RAG over the HR-policy PDFs, must respect employee privacy (no PII leaves the company), must be approachable by non-technical employees. You can choose the LLM provider but must justify it. Label all three tiers and the data flow.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example sketch (text form):**

```
  PRESENTATION                 BUSINESS LOGIC                 DATA
  ────────────                 ──────────────                 ────

  Streamlit chat UI            FastAPI service                Local vector store
  (internal URL,        ▶      (Python, one container)  ▶    (Chroma or FAISS,
   SSO required)               • RAG retrieval                same container or
                       ◀      • prompt assembly         ◀    sidecar)
                               • LLM call (Ollama local
                                 or Azure OpenAI with
                                 EU residency)                Source PDFs in S3
                               • response post-process        (nightly re-index
                                                              job, NB 40 pattern)
```

**Provider justification:** Ollama with `llama3.2:8b` *or* Azure OpenAI with the EU residency flag. The constraint *"no PII leaves the company"* rules out direct calls to OpenAI / Anthropic public endpoints from a HR context where employee names appear in queries. Ollama is the simplest local option; Azure with EU residency is the simplest enterprise-approved cloud option.

**Notable design choices:** SSO (not a custom auth — see Practice 5); a nightly re-index job rather than live updating (HR policies change weekly at most); the vector store co-located with the API (no separate network hop for retrieval); a Streamlit UI because non-technical employees are the audience and Streamlit's pre-built chat UI is well-understood.
</details>

### Stretch exercise B — ⭐⭐⭐ Cost a microservices migration

A 12-person engineering team is considering migrating their 3-tier Python monolith (50k LoC) to ~10 microservices. They argue *"it'll let us deploy independently and scale better."* List five engineering costs they will incur in the first 12 months that the proposal doesn't mention. For each, estimate the order of magnitude (engineer-days).

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Five hidden costs:**

1. **Distributed tracing & observability.** Going from "check the logs" to needing OpenTelemetry / Jaeger / Datadog APM is real. **~40–80 engineer-days** to instrument 10 services + maintain.

2. **Service-to-service contracts.** Schemas, versioning, breaking-change communication, contract tests. **~60 engineer-days** for the initial contracts; ongoing forever.

3. **Per-service CI/CD pipelines.** Build, test, deploy, rollback for each service. Even with templating, **~30 engineer-days** to set up + **~10 days/year/service** to maintain = ~130 days first year.

4. **Auth and security across services.** Token issuing, validation, secrets per service, network policies. **~30–60 engineer-days.**

5. **Eventual consistency & data ownership.** No more cross-table joins; data has to be denormalised across services or fetched via API. Bugs that the monolith would have caught at compile time now appear in production. **~50 engineer-days of refactoring + ongoing bug investigation.**

**Total order of magnitude:** ~300–400 engineer-days in year one. For a 12-person team that's about 15% of total capacity for the year, just on the migration's overhead — before any user-visible feature ships. The proposal needs to argue that the *deploy independence* and *scaling* benefits are worth that, in real numbers.
</details>

### Stretch exercise C — ⭐⭐⭐ Architecture review on a real-ish ADR

Pick a real piece of software you use (or a hypothetical one similar to your day job). Write a 1-page architecture decision record (ADR) for adding an AI feature to it. The ADR should contain:

1. *Context* (what is the problem we're solving?)
2. *Decision* (which architecture pattern, why)
3. *Consequences* (what we're giving up by picking this)
4. *Alternatives considered* (at least two)
5. *Out of scope* (things this ADR is **not** trying to solve)

ADRs are 1–2 pages, written for the next engineer who has to maintain the thing. The discipline of writing one before building is the single highest-leverage architectural practice that exists.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Sketch of an ADR for adding an AI-assisted code-review bot to an internal Git platform** (illustrative — yours can be anything):

> **ADR-014 — AI-assisted code review for internal Git server**
>
> *Context.* PR reviews take an average of 9 hours of wall-clock time to land. ~60% of comments are style / typo / obvious-bug pattern matches an LLM should catch. We want to reduce reviewer fatigue.
>
> *Decision.* Add a **single-tier scheduled job** that polls open PRs hourly, runs each diff through an LLM, and posts comments via the existing GitHub-Enterprise webhook. No new database; no new auth; no new UI. Configurable per-repo (opt-in, not on by default).
>
> *Consequences.* (+) ships in 1 sprint, no operational overhead. (+) deletable in 1 commit if it doesn't pay off. (–) polling is wasteful at scale (revisit if > 200 PRs/day). (–) no audit log beyond what GitHub already keeps.
>
> *Alternatives considered.* (a) Build it as a microservice with its own queue and a real DB → over-engineered for current scale. (b) Use a SaaS code-review tool → privacy review would take 4–6 months. (c) Buy GitHub Copilot for everyone instead → orthogonal, doesn't replace PR comments; consider separately.
>
> *Out of scope.* (a) Auto-approving PRs — humans always approve. (b) Style-guide enforcement — that's the linter's job. (c) Security scanning — owned by InfoSec team.

**What makes this good:** specific scale numbers, explicit alternatives + why-rejected, an explicit *out-of-scope* list (the most underused part of ADRs), and an architectural choice (scheduled job) that matches the team and the volume.
</details>

### Stretch exercise D — ⭐⭐⭐ Convert a notebook into a 3-tier sketch

Take **NB 42 — the AI-assistant capstone** (or NB 41 — analytics capstone). It's currently a single-tier notebook. Sketch how you'd promote it into a 3-tier system used by 30 customer-support agents inside a company. Be specific about:

1. What the presentation tier looks like (and which tool).
2. What stays in / what changes in the business-logic tier.
3. What the data tier looks like (and whether you need a vector store separate from the relational DB).
4. What you'd add for monitoring (cost, accuracy, latency dashboards).
5. Which parts of the notebook become a *script* that runs on a schedule rather than on demand.

This exercise is the bridge between *can build the AI feature* (the capstones prove that) and *can deploy it inside a company* (which this module is about).

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example sketch for NB 42 → production deployment:**

1. **Presentation tier.** Streamlit (or a thin React app embedded in the agents' existing CRM tab). Chat-style UI, shows the AI's suggested response and confidence; agents click 'send', 'edit', or 'reject'.

2. **Business-logic tier.** FastAPI service hosting: the prompt-assembly code from the capstone, retries / timeouts / cost-cap from NB 40, structured-output validation from NB 21. **What changes:** the `MockLLM` is swapped for the production provider via the `llm_providers.py` shim; the in-notebook `print` statements are replaced with structured `logging.info` going to the company's log aggregator.

3. **Data tier.** PostgreSQL for ticket metadata, agent actions, and audit logs. **Separately**: a vector store (Chroma or pgvector inside the same Postgres) for the RAG knowledge base — semantic queries shouldn't go through SQL.

4. **Monitoring.** A small dashboard (Streamlit or Grafana) tracking: daily token cost; daily accept / edit / reject rate; p50 / p95 response latency; per-agent adoption %. Alerts if cost > 1.5× rolling 7-day average or accept-rate drops > 10pp.

5. **Scheduled scripts.** (a) Nightly re-indexing of the RAG knowledge base when policy PDFs change (NB 40 pattern). (b) Weekly evaluation script — a held-out 'gold set' of 50 tickets run against the current model + prompt; results saved to a CSV that powers the accuracy panel of the dashboard (NB 25 pattern).

The exercise's point: this is *not* a re-write. About 70% of the notebook's code is reused verbatim in the business-logic tier; the remaining 30% is auth, persistence, error-handling, and monitoring. That's the actual shape of *productionising* a notebook.
</details>

## 🧠 Key takeaways

- Five canonical patterns: single-tier, 3-tier, service-oriented, microservices, ML pipeline.
- The right pattern is determined by *traffic*, *team size*, and *coordination cost* — not by what sounds modern.
- The default for everything is to start single-tier. Promote only when you have *evidence* of a bottleneck.
- Modular monolith is the underrated middle ground between 3-tier and microservices for most product teams.
- ML pipelines are a cross-cutting concern that sit on top of any of the above; their distinguishing feature is that the *model* is a versioned artefact and drift exists.

## ✅ Self-assessment

- I can draw all five patterns from memory and explain when each one is the right answer.
- I can recognise the smell of having outgrown an architecture, and the smell of having over-architected one.
- I can write a short ADR for an AI-feature decision, with explicit alternatives and out-of-scope.
- I can articulate what makes an ML pipeline different from a regular software pipeline.

## 🚀 Next step

→ **NB 45 — AI-assisted software development** (`./45_ai_assisted_software_development.ipynb`). Now that you can choose *what* to build and *how to structure* it, the next question is *how to actually build it efficiently in 2026* — modern IDEs, Git, prompt engineering for code, and the critical-review discipline that keeps AI-generated artefacts from silently introducing bugs.
